# Soony · Grand Heist 6 launch: data exploration

Exploratory plots for the Lio challenge **"Keep the Line Running"**. Everything below is derived from the portal data at runtime: no values are copied from the data into code. Re-run the notebook (Kernel → Restart & Run All) after new files land in the portal.

| # | Section | Question it answers |
|---|---------|---------------------|
| 1 | Launch readiness | How many hours do we need vs. have, per edition? |
| 2 | Pack-out | Are we packing fast enough? |
| 3 | Parts vs. demand | Which parts do not cover the remaining pack-out? |
| 4 | Inbound shipments | What arrives before the line stops? Where do sources disagree? |
| 5 | Supplier performance | Who is late, who promises later than requested? |
| 6 | Sourcing & prices | Single sources, price moves, spend |
| 7 | Goods receipts | Short / over deliveries, receiving remarks |
| 8 | Data quality | What is missing or contradictory? |

**Confidential:** portal data, use for this hackathon only.

In [ ]:
import os, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 12})
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 60)

def find_data_dir():
    """SOONY_DATA_DIR, else the nearest soony-data/ or data/ folder above the notebook."""
    cands = [Path(os.environ["SOONY_DATA_DIR"])] if os.environ.get("SOONY_DATA_DIR") else []
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        cands += [base / "soony-data", base / "data"]
    for c in cands:
        if (c / "soony").is_dir() and (c / "suppliers").is_dir():
            return c
    raise FileNotFoundError("Data folder not found. Set SOONY_DATA_DIR to the folder containing soony/ and suppliers/.")

DATA = find_data_dir()
print("data dir:", DATA)
print(f"{sum(1 for p in DATA.rglob('*') if p.is_file())} files · " +
      ", ".join(f"{p.name}: {sum(1 for q in p.rglob('*') if q.is_file())}" for p in sorted(DATA.iterdir()) if p.is_dir()))

## 0 · Load and normalise
Loaders glob the folders, so new files (new supplier folders, new PO exports) are picked up on the next run. The two SAP exports have different layouts (`;` separator, German field names, `MM/DD/YYYY`, decimal commas, numeric supplier ids), so they are mapped onto one schema.

In [ ]:
def read_csv_any(path, **kw):
    """Read a CSV, sniffing ; vs , as separator."""
    head = Path(path).read_text(encoding="utf-8-sig", errors="replace").splitlines()[0]
    sep = ";" if head.count(";") > head.count(",") else ","
    return pd.read_csv(path, sep=sep, **kw)

def read_sheet(path, sheet, marker):
    """Read an Excel sheet whose header row is somewhere below a title block: header = first row containing `marker`."""
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    idx = raw.index[raw.apply(lambda r: r.astype(str).str.strip().eq(marker).any(), axis=1)][0]
    df = raw.iloc[idx + 1:].copy()
    df.columns = [str(c).strip() for c in raw.iloc[idx]]
    return df.dropna(how="all").reset_index(drop=True)

# --- master data ---------------------------------------------------------------------------
M = DATA / "soony" / "master"
products   = pd.read_csv(M / "products.csv", parse_dates=["launch_date"])
components = pd.read_csv(M / "components.csv")
bom        = pd.read_csv(M / "bom.csv")
ports      = pd.read_csv(M / "ports.csv")
routes     = pd.read_csv(M / "routes.csv")

# --- suppliers (one folder each) -----------------------------------------------------------
profiles = pd.concat([pd.read_csv(p) for p in sorted(DATA.glob("suppliers/*/profile.csv"))], ignore_index=True)
supplier_name = dict(zip(profiles.supplier_id, profiles.name))
supplier_label = {k: f"{k[-2:]} {v}" for k, v in supplier_name.items()}
sup_components = pd.concat([pd.read_csv(p).assign(supplier_id=p.parent.name.split("_")[0])
                            for p in sorted(DATA.glob("suppliers/*/components.csv"))], ignore_index=True)
sup_currency = dict(zip(profiles.supplier_id, profiles.currency))
SUP_ORDER = sorted(supplier_name)
SUP_COLORS = dict(zip(SUP_ORDER, sns.color_palette("tab10", len(SUP_ORDER))))
comp_desc = dict(zip(components.component_id, components.description))
def short(cid, n=26):
    d = comp_desc.get(cid, "")
    return f"{cid.split('-',1)[1]} {d[:n]}"

# --- purchase orders: two SAP exports -> one schema ----------------------------------------
LEGACY_MAP = {"EBELN": "po_number", "EBELP": "po_line", "LIFNR": "supplier_id", "MATNR": "material_id", "WERKS": "plant",
              "MENGE": "quantity", "NETPR": "unit_price", "BEDAT": "order_date", "EINDT": "requested_date",
              "ZZCONF": "promised_date", "WEDAT": "actual_delivery_date", "ZZSTAT": "status", "ERNAM": "buyer"}
frames = []
for p in sorted((DATA / "soony" / "erp").glob("purchase_orders*.csv")):
    df = read_csv_any(p, dtype=str)
    if "EBELN" in df.columns:  # legacy layout
        df = df.rename(columns=LEGACY_MAP)
        df["supplier_id"] = "SUP-" + df["supplier_id"].str.strip().str.zfill(4)
        df["unit_price"] = df["unit_price"].str.replace(",", ".", regex=False)
        for c in ["order_date", "requested_date", "promised_date", "actual_delivery_date"]:
            df[c] = pd.to_datetime(df[c], format="%m/%d/%Y", errors="coerce")
        df["currency"] = df["supplier_id"].map(sup_currency)
    else:
        for c in ["order_date", "requested_date", "promised_date", "actual_delivery_date"]:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    df["po_line"] = pd.to_numeric(df["po_line"])
    df["source_file"] = p.name
    frames.append(df)
po = pd.concat(frames, ignore_index=True)
for c in ["quantity", "unit_price"]:
    po[c] = pd.to_numeric(po[c])
po["po_number"] = po["po_number"].astype(str)
po["supplier"] = po["supplier_id"].map(supplier_label)
po["promised_vs_requested"] = (po.promised_date - po.requested_date).dt.days
po["delivered_vs_promised"] = (po.actual_delivery_date - po.promised_date).dt.days
po["delivered_vs_requested"] = (po.actual_delivery_date - po.requested_date).dt.days
po["lead_time_days"] = (po.actual_delivery_date - po.order_date).dt.days

# --- other ERP -----------------------------------------------------------------------------
E = DATA / "soony" / "erp"
gr     = pd.read_csv(E / "goods_receipts.csv", parse_dates=["gr_date"], dtype={"po_number": str})
stock  = pd.read_csv(E / "stock_snapshot.csv", parse_dates=["snapshot_date", "last_count"])
packed = pd.read_csv(E / "packout_log.csv", parse_dates=["date"])
ship   = pd.read_csv(E / "open_shipments.csv", parse_dates=["etd", "eta"])
ship["updated_at"] = pd.to_datetime(ship["updated_at"], utc=True).dt.tz_convert("Europe/Berlin").dt.tz_localize(None)
ship["supplier"] = ship.supplier_id.map(supplier_label)

# --- launch plan + manual tracker ----------------------------------------------------------
LP = DATA / "soony" / "context" / "launch_plan_GH6.xlsx"
lp_editions = read_sheet(LP, "Editions", "Edition")
lp_parts    = read_sheet(LP, "Parts", "Part")
lp_inbound  = read_sheet(LP, "Inbound", "Supplier")
lp_shifts   = read_sheet(LP, "Shifts", "Day")
for df, cols in [(lp_editions, lp_editions.columns[2:]), (lp_parts, lp_parts.columns[2:8]), (lp_inbound, ["Quantity"])]:
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
lp_inbound["ETA"] = pd.to_datetime(lp_inbound["ETA"])

tracker_path = next((DATA / "soony" / "logistics").glob("*.xlsx"))
tracker = read_sheet(tracker_path, 0, "Shipment")
tracker[["ETD", "ETA"]] = tracker[["ETD", "ETA"]].apply(pd.to_datetime)

# --- reference times, all derived ---------------------------------------------------------
NOW = ship.updated_at.max()
launch_day = products.loc[products.lifecycle.eq("launch"), "launch_date"].max()
last_shift_end = lp_shifts["Shift"].dropna().str.extract(r"–\s*(\d{1,2}):(\d{2})").astype(float).dropna()
DEADLINE = launch_day + pd.Timedelta(hours=last_shift_end.iloc[-1, 0], minutes=last_shift_end.iloc[-1, 1])
edition_name = dict(zip(products.product_id, products.name.str.replace("Grand Heist 6 – ", "", regex=False)))
print(f"now (latest portal update): {NOW:%a %d %b %H:%M} · packing ends / trucks leave: {DEADLINE:%a %d %b %H:%M}"
      f" · {(DEADLINE - NOW).total_seconds() / 3600:.0f} h to go")
print(f"{len(po)} PO lines from {po.source_file.nunique()} exports · {len(gr)} goods receipts · {len(ship)} open shipments · {len(profiles)} suppliers")

## 1 · Launch readiness: hours needed vs. hours available
Taken from Soony's own launch plan. The Collector's cell is the tight one: any hour lost to a missing part costs a full hour of its rate.

In [ ]:
ed = lp_editions.copy()
ed["label"] = ed["Product"].map(edition_name).fillna(ed["Edition"])
ed["util"] = ed["Hours needed"] / ed["Hours available"]
still = ed.set_index("Product")["Still to pack"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
x = np.arange(len(ed)); w = 0.38
axes[0].bar(x - w/2, ed["Packed"], w, label="Packed", color="#2a9d8f")
axes[0].bar(x + w/2, ed["Still to pack"], w, label="Still to pack", color="#e76f51")
axes[0].set_xticks(x, ed["Edition"]); axes[0].set_title("Day-one demand: packed vs. still to pack")
axes[0].yaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k"); axes[0].legend()
for i, r in ed.iterrows():
    axes[0].text(i - w/2, r["Packed"], f"{r['Packed']/r['Day-one demand']:.0%}", ha="center", va="bottom", fontsize=9)
    axes[0].text(i + w/2, r["Still to pack"], f"{r['Still to pack']:,.0f}", ha="center", va="bottom", fontsize=9)

axes[1].barh(ed["Edition"], ed["Hours available"], color="#e5e5e5", label="Hours available")
axes[1].barh(ed["Edition"], ed["Hours needed"], color=["#e76f51" if u > 0.9 else "#2a9d8f" for u in ed["util"]], height=0.45, label="Hours needed")
for i, r in ed.iterrows():
    axes[1].text(r["Hours needed"] + 0.3, i, f"{r['Hours needed']:.1f} h of {r['Hours available']:.0f} h ({r['util']:.0%}) · {r['Rate per hour']:,.0f}/h", va="center", fontsize=9)
axes[1].set_title("Line utilisation until launch (red above 90 %)"); axes[1].set_xlim(0, ed["Hours available"].max() * 1.6)
plt.tight_layout(); plt.show()

slack = (ed["Hours available"] - ed["Hours needed"]) * ed["Rate per hour"]
print("Slack in units if every part is there:\n" + slack.set_axis(ed["Edition"]).round(0).astype(int).to_string())

## 2 · Pack-out progress
Daily output per line, cumulative output, and the rate we would still need to finish in the remaining hours.

In [ ]:
pk = packed.copy()
pk["edition"] = pk["product_id"].map(edition_name)
daily = pk.pivot_table(index="date", columns="edition", values="units_packed", aggfunc="sum").fillna(0)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
daily.plot(ax=axes[0], marker="o", ms=3); axes[0].set_title("Units packed per working day")
axes[0].yaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k"); axes[0].set_xlabel("")
daily.cumsum().plot.area(ax=axes[1], alpha=0.75, stacked=False); axes[1].set_title("Cumulative units packed")
axes[1].yaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k"); axes[1].set_xlabel("")

# required output per shift day if the remainder is spread over the remaining shifts (weighted by shift hours) vs. last logged day
shifts = lp_shifts.dropna(subset=["Day"]).copy(); shifts["Hours"] = shifts["Hours"].astype(float)
last_day = pk[pk.date == pk.date.max()].groupby("product_id")["units_packed"].sum()
rows = {}
for pid, rem in still.items():
    rows[edition_name.get(pid, pid)] = {f"Last logged day ({pk.date.max():%d %b})": last_day.get(pid, 0),
                                       **{f"Needed on {r.Day} ({r.Hours:.0f} h)": rem * r.Hours / shifts.Hours.sum() for r in shifts.itertuples()}}
pd.DataFrame(rows).T.plot.bar(ax=axes[2], rot=0, color=["#8d99ae", "#e76f51", "#f4a261"])
axes[2].set_title("Last day's output vs. needed per remaining shift day"); axes[2].set_xlabel(""); axes[2].legend(fontsize=8)
axes[2].yaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k")
plt.tight_layout(); plt.show()

## 3 · Do the parts cover the remaining pack-out?
Parts needed = BOM × units still to pack (computed here, not read from the plan). Stock is from the ERP snapshot, which the plan itself labels "not counted". We then add what the launch plan says is due before launch.

In [ ]:
need = (bom[bom.parent_id.isin(still.index)]
        .assign(units=lambda d: d.parent_id.map(still) * d.qty_per)
        .groupby("child_id")["units"].sum())
snap = stock[stock.snapshot_date == stock.snapshot_date.max()]
stk = snap[~snap.material_id.str.startswith("FG-")].groupby("material_id").agg(
    on_hand=("quantity", "sum"), safety=("safety_stock", "first"), weeks_cover=("weeks_of_cover", "first"), last_count=("last_count", "first"))
due = lp_inbound[lp_inbound["ETA"] <= DEADLINE].groupby("Part")["Quantity"].sum()
parts = (pd.DataFrame({"need": need}).join(stk, how="outer").join(due.rename("inbound_due")).fillna({"inbound_due": 0, "need": 0, "on_hand": 0}))
parts["inbound_due"] = parts["inbound_due"].where(parts.need > 0, 0)
parts = parts[parts.need > 0].copy()
parts["gap_stock_only"] = parts.on_hand - parts.need
parts["gap_with_inbound"] = parts.on_hand + parts.inbound_due - parts.need
parts["label"] = [short(i) for i in parts.index]
parts = parts.sort_values("gap_with_inbound")

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1.5, 1]})
y = np.arange(len(parts))
axes[0].barh(y, parts.need, color="#e5e5e5", label="Needed for remaining pack-out")
axes[0].barh(y, parts.on_hand, height=0.5, color="#457b9d", label="On hand (ERP)")
axes[0].barh(y, parts.inbound_due, left=parts.on_hand, height=0.5, color="#a8dadc", label="Inbound, ETA before deadline (plan)")
axes[0].set_yticks(y, parts.label); axes[0].xaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k")
axes[0].set_title("Need vs. supply per part"); axes[0].legend(loc="lower right", fontsize=8)
axes[1].barh(y, parts.gap_with_inbound, color=["#e76f51" if g < 0 else "#2a9d8f" for g in parts.gap_with_inbound])
axes[1].set_yticks(y, [""] * len(y)); axes[1].axvline(0, color="k", lw=0.8)
axes[1].set_title("Surplus / shortfall (stock + inbound − need)")
axes[1].xaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k")
for i, g in enumerate(parts.gap_with_inbound):
    axes[1].text(g, i, f" {g:,.0f}", va="center", ha="left" if g >= 0 else "right", fontsize=8)
plt.tight_layout(); plt.show()

# compare our calculation against the plan's own "Cover" column: where do they disagree?
chk = parts.join(lp_parts.set_index("Part")[["Needed for the remaining pack-out", "ERP stock", "Cover", "Status"]])
chk["Δ need vs plan"] = chk.need - chk["Needed for the remaining pack-out"]
chk["Δ stock vs plan"] = chk.on_hand - chk["ERP stock"]
chk[["need", "on_hand", "inbound_due", "gap_stock_only", "gap_with_inbound", "Cover", "Status", "Δ need vs plan", "Δ stock vs plan"]].round(0)

In [ ]:
# Stock position vs. safety stock and weeks of cover for every part, plus how stale the count is
s = stk.copy(); s["label"] = [short(i) for i in s.index]
s["stock_vs_safety"] = s.on_hand / s.safety.replace(0, np.nan)
s["count_age_days"] = (stock.snapshot_date.max() - s.last_count).dt.days
s = s.sort_values("stock_vs_safety")
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), sharey=True)
axes[0].barh(s.label, s.stock_vs_safety.fillna(0), color=["#e76f51" if v < 1 else "#2a9d8f" for v in s.stock_vs_safety.fillna(0)])
axes[0].axvline(1, color="k", ls="--", lw=0.8); axes[0].set_title("On hand ÷ safety stock (red < 1)")
axes[1].barh(s.label, s.weeks_cover, color="#457b9d"); axes[1].set_title("Weeks of cover (ERP)")
axes[2].barh(s.label, s.count_age_days, color="#8d99ae"); axes[2].set_title("Days since last stock count")
plt.tight_layout(); plt.show()

**Parts with no stock at all** (e.g. magnets for the Collector's lid) and parts that are needed but have no open PO deserve a separate look:

In [ ]:
open_po = po[po.status.isin(["open", "awaiting_approval"])]
cover = parts[["need", "on_hand", "inbound_due"]].join(open_po.groupby("material_id").quantity.sum().rename("open_po_qty")).fillna(0)
cover["has_supplier"] = cover.index.isin(sup_components.component_id)
cover["has_open_po"] = cover.open_po_qty > 0
cover["label"] = [short(i) for i in cover.index]
cover[(cover.on_hand + cover.open_po_qty < cover.need)]

## 4 · Inbound shipments
### 4a · Timeline of all open shipments against the packing deadline

In [ ]:
sh = ship.sort_values(["eta", "etd"]).reset_index(drop=True)
status_colors = {"arrived_port": "#2a9d8f", "in_delivery": "#2a9d8f", "in_transit": "#457b9d", "booked": "#a8dadc", "delayed": "#e76f51"}
fig, ax = plt.subplots(figsize=(13, 0.32 * len(sh) + 1.5))
for i, r in sh.iterrows():
    c = status_colors.get(r.status, "#999")
    ax.plot([r.etd, r.eta], [i, i], color=c, lw=5, solid_capstyle="round", alpha=0.9)
    ax.plot(r.eta, i, "o", color=c, ms=7, mec="k", mew=0.5)
late = sh.eta > DEADLINE
ax.set_yticks(range(len(sh)), [f"{r.shipment_id[-5:]} {r.mode:<4} {r.supplier.split(' ', 1)[1][:18]:<18} {r.material_ids.replace('CMP-', '')[:14]}" for r in sh.itertuples()], fontsize=8, family="monospace")
ax.axvline(NOW, color="k", ls=":", lw=1); ax.text(NOW, -0.9, "now ", fontsize=8, va="bottom", ha="right")
ax.axvline(DEADLINE, color="#d00000", lw=2); ax.text(DEADLINE, -0.9, " packing ends", color="#d00000", fontsize=8, va="bottom", ha="left")
ax.axvspan(DEADLINE, sh.eta.max() + pd.Timedelta(days=3), color="#d00000", alpha=0.06)
ax.invert_yaxis(); ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b")); ax.set_title(f"Open shipments, ETD → ETA ({late.sum()} of {len(sh)} arrive after the deadline)")
ax.legend(handles=[plt.Line2D([0], [0], color=c, lw=5, label=k) for k, c in status_colors.items()], loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ### 4b · Shipments by supplier and arrival window; map of last known positions
win = pd.cut((ship.eta - DEADLINE).dt.total_seconds() / 3600, [-np.inf, 0, 72, np.inf], labels=["before deadline", "0–3 days late", "> 3 days late"])
mix = pd.crosstab(ship.supplier, win).reindex(columns=["before deadline", "0–3 days late", "> 3 days late"], fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 1.5]})
mix.plot.barh(stacked=True, ax=axes[0], color=["#2a9d8f", "#f4a261", "#e76f51"]); axes[0].set_title("Open shipments by supplier vs. deadline"); axes[0].set_ylabel("")
ax = axes[1]
hub = ports.set_index("locode")
for r in routes.itertuples():
    if r.origin in hub.index and r.destination in hub.index:
        a, b = hub.loc[r.origin], hub.loc[r.destination]
        ax.plot([a.lon, b.lon], [a.lat, b.lat], color="#ccc", lw=0.8, zorder=1)
ax.scatter(ports.lon, ports.lat, s=25, color="k", zorder=3)
for p in ports.itertuples():
    ax.annotate(p.locode, (p.lon, p.lat), fontsize=7, xytext=(3, 3), textcoords="offset points")
for st, g in ship.groupby("status"):
    ax.scatter(g.last_lon, g.last_lat, s=70, color=status_colors.get(st, "#999"), edgecolor="k", lw=0.5, label=st, zorder=4)
ax.scatter(profiles.lon, profiles.lat, marker="s", s=60, color=[SUP_COLORS[s] for s in profiles.supplier_id], edgecolor="k", zorder=5)
ax.set_title("Last known positions (dots), ports (black), supplier sites (squares)"); ax.legend(fontsize=8, loc="lower left"); ax.set_xlabel("lon"); ax.set_ylabel("lat")
plt.tight_layout(); plt.show()

### 4c · Where do the sources disagree?
Three views of the same shipments exist: the ERP/carrier feed (`open_shipments.csv`), the shared Excel tracker, and the launch plan's inbound sheet. We surface the disagreements instead of picking a side.

In [ ]:
trk = tracker.rename(columns={"Shipment": "shipment_id", "ETA": "eta_tracker", "Status": "status_tracker", "Comment": "tracker_comment"})
cmp_s = ship.merge(trk[["shipment_id", "eta_tracker", "status_tracker", "tracker_comment"]], on="shipment_id", how="outer", indicator=True)
cmp_s["eta_diff_days"] = (cmp_s.eta_tracker - cmp_s.eta).dt.days
cmp_s["status_tracker_norm"] = cmp_s.status_tracker.str.replace(" ", "_")
cmp_s["status_conflict"] = cmp_s.status.notna() & cmp_s.status_tracker.notna() & (cmp_s.status != cmp_s.status_tracker_norm)

in_tracker_only = (cmp_s._merge == "right_only").sum(); in_erp_only = (cmp_s._merge == "left_only").sum()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(["in both", "ERP feed only\n(missing in tracker)", "tracker only"], [(cmp_s._merge == "both").sum(), in_erp_only, in_tracker_only], color=["#8d99ae", "#e76f51", "#f4a261"])
axes[0].set_title("Shipment coverage: feed vs. Excel tracker")
cmp_s[cmp_s._merge == "left_only"].groupby("mode").size().reindex(["sea", "air", "road"], fill_value=0).plot.bar(ax=axes[1], color="#e76f51", rot=0)
axes[1].set_title("…and the missing ones by mode"); axes[1].set_xlabel("")
both = cmp_s[cmp_s._merge == "both"]
st = pd.DataFrame({"ERP feed": both.status.value_counts(), "Excel tracker": both.status_tracker_norm.value_counts()}).fillna(0)
st.plot.bar(ax=axes[2], rot=30, color=["#457b9d", "#f4a261"]); axes[2].set_title(f"Status of the {len(both)} shared shipments, by source")
plt.tight_layout(); plt.show()
cmp_s[cmp_s.status_conflict | (cmp_s.eta_diff_days.fillna(0) != 0)][["shipment_id", "supplier", "status", "status_tracker", "eta", "eta_tracker", "tracker_comment", "note"]]

In [ ]:
# Plan's inbound sheet vs. feed: same container/AWB, different ETA or status?
pin = lp_inbound.rename(columns={"Reference": "reference", "ETA": "eta_plan", "Status": "status_plan"})
x = pin.merge(ship[["reference", "eta", "status", "supplier_id"]], on="reference", how="left")
x["eta_diff_days"] = (x.eta_plan - x.eta).dt.days
x[["Supplier", "Part", "Quantity", "Mode", "reference", "eta_plan", "eta", "eta_diff_days", "status_plan", "status"]]

## 5 · Supplier performance
### 5a · Delivery reliability from purchase orders (both exports, delivered lines only)

In [ ]:
dl = po[(po.status.isin(["delivered", "GL"])) & po.actual_delivery_date.notna() & (po.quantity > 0)].copy()
order = [supplier_label[s] for s in SUP_ORDER if supplier_label[s] in set(dl.supplier)]
pal = {supplier_label[s]: SUP_COLORS[s] for s in SUP_ORDER}

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
sns.boxplot(data=dl, y="supplier", x="delivered_vs_promised", order=order, palette=pal, ax=axes[0], fliersize=2)
axes[0].axvline(0, color="k", lw=0.8); axes[0].set_title("Days late vs. promised date"); axes[0].set_ylabel(""); axes[0].set_xlabel("days (> 0 = late)")
ontime = dl.groupby("supplier").agg(on_time=("delivered_vs_promised", lambda s: (s <= 0).mean()),
                                    on_time_vs_requested=("delivered_vs_requested", lambda s: (s <= 0).mean())).reindex(order)
ontime.rename(columns={"on_time": "vs. promised", "on_time_vs_requested": "vs. requested"}).plot.barh(ax=axes[1], color=["#2a9d8f", "#e76f51"], width=0.75)
axes[1].set_xlim(0, 1.25); axes[1].set_yticklabels([]); axes[1].set_ylabel(""); axes[1].legend(fontsize=8, loc="upper right")
axes[1].set_xticks([0, .25, .5, .75, 1])
axes[1].xaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); axes[1].set_title("Share of lines delivered on time")
dl["month"] = dl.actual_delivery_date.dt.to_period("Q").dt.to_timestamp()
trend = dl.groupby(["month", "supplier"])["delivered_vs_promised"].mean().unstack().reindex(columns=order)
trend.plot(ax=axes[2], marker="o", ms=3, color=[pal[c] for c in trend.columns], legend=False); axes[2].axhline(0, color="k", lw=0.8)
axes[2].set_title("Mean days late vs. promised, by quarter"); axes[2].set_xlabel(""); axes[2].legend(order, fontsize=7, loc="upper right")
plt.tight_layout(); plt.show()

In [ ]:
# ### 5b · Promised vs. requested: do suppliers commit to a later date than Soony asked for? (open + delivered)
po_ok = po[po.quantity > 0]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.4))
sns.boxplot(data=po_ok, y="supplier", x="promised_vs_requested", order=order, palette=pal, ax=axes[0], fliersize=2)
axes[0].axvline(0, color="k", lw=0.8); axes[0].set_title("Promised date − requested date (days)"); axes[0].set_ylabel("")
op = po_ok[po_ok.status.eq("open")].copy()
op["late_flag"] = op.promised_date > DEADLINE
q = op.groupby(["supplier", "late_flag"]).quantity.sum().unstack(fill_value=0).reindex(columns=[False, True], fill_value=0).reindex(order).fillna(0)
q.columns = ["promised before deadline", "promised after deadline"]
q.plot.barh(stacked=True, ax=axes[1], color=["#2a9d8f", "#e76f51"]); axes[1].set_ylabel("")
axes[1].xaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k"); axes[1].set_title("Open PO quantity by promised date vs. packing deadline")
plt.tight_layout(); plt.show()

In [ ]:
# ### 5c · Observed lead time vs. the lead time in the supplier's own component list
lt_obs = dl.groupby(["supplier_id", "material_id"]).lead_time_days.agg(["median", "max", "count"]).reset_index()
lt = lt_obs.merge(sup_components[["supplier_id", "component_id", "lead_time_days", "primary"]], left_on=["supplier_id", "material_id"], right_on=["supplier_id", "component_id"], how="left")
lt["label"] = [f"{s[-2:]} · {m.split('-', 1)[1]}" for s, m in zip(lt.supplier_id, lt.material_id)]
lt = lt.sort_values(["supplier_id", "material_id"]).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(12, 0.3 * len(lt) + 1.5))
ax.barh(lt.label, lt["median"], color=[SUP_COLORS[s] for s in lt.supplier_id], alpha=0.85, label="Observed median (order → delivery)")
ax.scatter(lt.lead_time_days, lt.label, color="k", marker="|", s=220, zorder=3, label="Stated lead time (supplier file)")
ax.scatter(lt["max"], lt.label, color="#d00000", marker=".", s=30, zorder=3, label="Observed max")
ax.invert_yaxis(); ax.set_xlabel("days"); ax.set_title("Lead time: stated vs. observed"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6 · Sourcing and prices
### 6a · Who can supply what? Share by supplier and component (single-source risk)

In [ ]:
sh_mat = sup_components.pivot_table(index="component_id", columns="supplier_id", values="share", aggfunc="sum").reindex(index=components.component_id)
sh_mat.index = [short(i, 30) for i in sh_mat.index]
sh_mat.columns = [supplier_label.get(c, c) for c in sh_mat.columns]
n_src = sh_mat.notna().sum(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2.2, 1]})
sns.heatmap(sh_mat, annot=True, fmt=".0%", cmap="Blues", vmin=0, vmax=1, cbar=False, linewidths=0.5, ax=axes[0], mask=sh_mat.isna())
axes[0].set_title("Share of volume by supplier (blank = not a supplier of that part)"); axes[0].set_ylabel("")
axes[1].barh(n_src.index, n_src, color=["#e76f51" if v < 2 else "#2a9d8f" for v in n_src]); axes[1].set_yticklabels([])
axes[1].set_title("Number of sources (red = single source)"); axes[1].set_xticks(range(0, int(n_src.max()) + 1)); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()
print("Parts used in the product BOMs without any supplier file:", sorted(set(bom.child_id) - set(sup_components.component_id)) or "none")

In [ ]:
# ### 6b · Price development per supplier/part (native currency, indexed to the first order) and spend
pp = po_ok.sort_values("order_date").copy()
pp["series"] = pp.supplier_id.str[-2:] + " · " + pp.material_id.str.split("-", n=1).str[1]
pp["first_price"] = pp.groupby("series").unit_price.transform("first")
pp["price_index"] = pp.unit_price / pp.first_price * 100
moving = pp.groupby("series").price_index.agg(lambda s: s.max() - s.min()).loc[lambda s: s > 0.5].index
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
for sname, g in pp[pp.series.isin(moving)].groupby("series"):
    axes[0].step(g.order_date, g.price_index, where="post", label=sname, alpha=0.9)
axes[0].axhline(100, color="k", lw=0.6); axes[0].set_title("PO unit price index (first order = 100), series with price moves"); axes[0].legend(fontsize=7, ncol=2); axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
pp["value"] = pp.quantity * pp.unit_price
spend = pp.groupby(["supplier", "currency"]).value.sum().unstack(fill_value=0).reindex(order)
spend.plot.barh(stacked=True, ax=axes[1]); axes[1].set_ylabel(""); axes[1].set_title("PO value by supplier (native currency, not converted)")
axes[1].xaxis.set_major_formatter(lambda v, _: f"{v/1e6:.1f}M")
plt.tight_layout(); plt.show()

### 6c · Open purchase-order book

In [ ]:
book = po[po.status.isin(["open", "awaiting_approval"])].copy()
book["promised_week"] = book.promised_date.dt.to_period("W").dt.start_time
piv = book.pivot_table(index="promised_week", columns="material_id", values="quantity", aggfunc="sum").fillna(0)
piv.columns = [short(c, 18) for c in piv.columns]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.4), gridspec_kw={"width_ratios": [2, 1]})
piv.plot.bar(stacked=True, ax=axes[0], width=0.85, colormap="tab20")
axes[0].set_xticklabels([d.strftime("%d %b") for d in piv.index], rotation=45); axes[0].set_xlabel("promised week (start)")
axes[0].set_title("Open PO quantity by promised week"); axes[0].legend(fontsize=7, ncol=2); axes[0].yaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k")
book.groupby(["supplier", "status"]).size().unstack(fill_value=0).reindex(order).plot.barh(stacked=True, ax=axes[1], color=["#f4a261", "#457b9d"]); axes[1].set_ylabel("")
axes[1].set_title("Open PO lines by status")
plt.tight_layout(); plt.show()
print("Waiting on approval:"); book[book.status == "awaiting_approval"][["po_number", "supplier", "material_id", "quantity", "unit_price", "currency", "promised_date", "note"]]

## 7 · Goods receipts
Receiving books what is on the paper, so short and over deliveries show up here first.

In [ ]:
g = gr.copy()
g["supplier"] = g.supplier_id.map(supplier_label)
g["fill"] = g.qty_received / g.qty_ordered
g["delta"] = g.qty_received - g.qty_ordered
g["remark_clean"] = g.remark.fillna("(none)")
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), gridspec_kw={"width_ratios": [1.2, 1.2, 1]})
clip_hi = 2.0
sns.stripplot(data=g.assign(fill_c=g.fill.clip(upper=clip_hi)), y="supplier", x="fill_c", order=order, palette=pal, ax=axes[0], size=3.5, alpha=0.7, jitter=0.25)
for r in g[g.fill > clip_hi].itertuples():
    axes[0].annotate(f"{r.fill:.0f}×", (clip_hi, order.index(r.supplier)), xytext=(-4, 6), textcoords="offset points", color="#d00000", fontsize=9, ha="right")
axes[0].axvline(1, color="k", lw=0.8); axes[0].set_title("Received ÷ ordered per receipt (clipped at 200 %)"); axes[0].set_ylabel(""); axes[0].set_xlabel("")
axes[0].xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
rc = g.remark_clean.str.slice(0, 34).value_counts().drop("(none)", errors="ignore").sort_values()
rc.plot.barh(ax=axes[1], color="#8d99ae"); axes[1].set_title(f"Receiving remarks ({g.remark.isna().sum()} receipts have none)"); axes[1].set_ylabel("")
g["outcome"] = np.select([g.fill < 1, g.fill > 1], ["short", "over"], "exact")
oc = pd.crosstab(g.supplier, g.outcome).reindex(order).reindex(columns=["short", "exact", "over"], fill_value=0)
oc.plot.barh(stacked=True, ax=axes[2], color=["#e76f51", "#2a9d8f", "#f4a261"]); axes[2].set_ylabel(""); axes[2].set_yticklabels([])
axes[2].set_title("Receipts by outcome"); axes[2].legend(fontsize=8, loc="upper right")
plt.tight_layout(); plt.show()

# Delivery notes are scanned at the dock (receiving/). Which receipts have a scan whose filename carries the delivery-note number?
scans = [p.name for p in (DATA / "soony" / "receiving").rglob("*") if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".pdf"}]
g["has_scan"] = [any(str(dn) in f for f in scans) for dn in g.delivery_note.fillna("")]
g["month"] = g.gr_date.dt.to_period("M").dt.to_timestamp()
cov = g.groupby("month").has_scan.agg(["sum", "count"]).assign(no_scan=lambda d: d["count"] - d["sum"])
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.bar(cov.index, cov["sum"], width=20, color="#2a9d8f", label="receipt with scanned delivery note")
ax.bar(cov.index, cov.no_scan, width=20, bottom=cov["sum"], color="#d9d9d9", label="no scan on the portal (yet)")
ax.set_title(f"Goods receipts by month and scan availability ({int(g.has_scan.sum())} of {len(g)} receipts have a scan)"); ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y")); plt.tight_layout(); plt.show()
print("Receipts without a PO reference or supplier:"); g[g.po_number.isna() | g.supplier_id.isna()][["gr_number", "gr_date", "material_id", "qty_received", "delivery_note", "remark"]]

## 8 · Data quality: what is missing or contradictory?

In [ ]:
def na_share(df, cols=None):
    cols = cols or df.columns
    return df[cols].isna().mean()

derived = {"supplier", "promised_vs_requested", "delivered_vs_promised", "delivered_vs_requested", "lead_time_days", "source_file"}
dq = {"PO lines": po.drop(columns=list(derived & set(po.columns))), "Goods receipts": gr, "Stock snapshot": stock, "Open shipments": ship.drop(columns=["supplier"]),
      "Supplier components": sup_components, "Supplier profiles": profiles}
tab = pd.DataFrame({k: na_share(v) for k, v in dq.items()}).T
tab = tab.loc[:, tab.max() > 0]
fig, axes = plt.subplots(1, 2, figsize=(16, 4.4), gridspec_kw={"width_ratios": [2.2, 1]})
sns.heatmap(tab, annot=True, fmt=".0%", cmap="Reds", vmin=0, vmax=1, cbar=False, linewidths=0.5, ax=axes[0]); axes[0].set_title("Share of missing values per field (only fields with gaps)")
issues = {
    "Receipts without PO ref": int(gr.po_number.isna().sum()),
    "Parts with zero stock": int((stk.on_hand == 0).sum()),
    "Tracker ≠ feed (status)": int(cmp_s.status_conflict.sum()),
    "Tracker ≠ feed (ETA)": int((cmp_s.eta_diff_days.fillna(0) != 0).sum()),
    "Shipments missing in tracker": int(in_erp_only),
    "Plan 'ok' but short (our calc)": int(((chk.Status.astype(str).str.lower() == "ok") & (chk.gap_with_inbound < 0)).sum()),
    "BOM parts w/o supplier file": len(set(bom.child_id) - set(sup_components.component_id)),
    "Open PO promised after deadline": int((book.promised_date > DEADLINE).sum()),
}
pd.Series(issues).sort_values().plot.barh(ax=axes[1], color="#e76f51"); axes[1].set_title("Findings to follow up")
for i, v in enumerate(pd.Series(issues).sort_values()):
    axes[1].text(v, i, f" {v}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Unstructured material that still needs reading (PDF scans, notices in KR / CN / PL): inventory by supplier and type
inv = pd.DataFrame([{"supplier": p.parts[p.parts.index("suppliers") + 1] if "suppliers" in p.parts else "soony", "type": p.parent.name, "ext": p.suffix.lower()}
                    for p in DATA.rglob("*") if p.is_file() and p.suffix.lower() in {".pdf", ".xlsx", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".md"}])
ct = pd.crosstab(inv.supplier.str.replace("_", " ").str.replace("SUP-00", ""), inv.type)
ct.plot.barh(stacked=True, figsize=(12, 4), colormap="Set2"); plt.title("Documents on the portal by supplier and folder (new files appear here after a re-run)")
plt.xlabel("files"); plt.tight_layout(); plt.show()

### Takeaways from this pass
Printed from the data, not typed in, so they stay correct when the data changes.

In [ ]:
tight = ed.sort_values("util").iloc[-1]
short_parts = parts[parts.gap_with_inbound < 0]
print(f"• {tight['Edition']} line needs {tight['Hours needed']:.1f} of {tight['Hours available']:.0f} h ({tight['util']:.0%}); each lost hour = {tight['Rate per hour']:,.0f} units.")
print(f"• {len(short_parts)} part(s) do not cover the remaining pack-out even with inbound due before the deadline: " +
      (", ".join(f"{i} ({v:,.0f})" for i, v in short_parts.gap_with_inbound.items()) or "none"))
print(f"• {int((ship.eta > DEADLINE).sum())} of {len(ship)} open shipments have an ETA after the packing deadline; {int((ship.status == 'delayed').sum())} are flagged delayed.")
print(f"• {in_erp_only} feed shipment(s) are missing from the Excel tracker; {int(cmp_s.status_conflict.sum())} status conflict(s), {int((cmp_s.eta_diff_days.fillna(0) != 0).sum())} ETA conflict(s).")
print(f"• Worst on-time rate vs. promised date: {ontime.on_time.idxmin()} ({ontime.on_time.min():.0%}).")
print(f"• {int((n_src < 2).sum())} part(s) have a single source.")